# 06 · Calculate MD work of adhesion

## Objective

The aim is to calculate the work of adhesion for a LiNbO3/SiO2 interface with the LiNbOSi-finetuned model. 

The notebook uses `MDAdhesionEnergyMaker` to run NVT MD for the top slab, bottom slab, and interface. Report the work of adhesion per unit area.

This notebook uses a short production run and skips equilibration. These settings show the workflow. They do not give a validated physical result. Geometry optimisation does not establish thermal equilibration. This is a demo of an MD calculation.

## Prerequisites

Before execution, make sure the following dependencies are met.

1. Install the notebook and MLIP dependencies:

   ```bash
   uv sync --group notebooks --group mlip
   ```

2. Set `[programs].mlip_models_dir` in `crystal.toml` to `mlip_models` directory. The `LiNbOSi-finetuned` files must be available.

The default input is the interface generated with Notebook 04. You can start this notebook independently or select the optimised interface from Notebook 05. No DFT software or database is required.

## 1. Import the MD workflow tools

This step imports the existing adhesion maker, MD settings, and result type. It imports the storage tools for local execution and the reader for the interface file.

In [ ]:
from ase.io import read
from jobflow.core.job import Job
from jobflow.core.store import JobStore
from jobflow.managers.local import run_locally
from maggma.stores import MemoryStore
from monty.json import jsanitize
from monty.serialization import dumpfn
from pymatgen.io.ase import AseAtomsAdaptor

from crystal import CRYSTAL_ROOT
from crystal.flows.md_adhesion_workflow_maker import MDAdhesionEnergyMaker
from crystal.schemas.md.adhesion import (
    MDAdhesionEnergyResult,
    MDAdhesionEnergySimulationSettings,
)
from crystal.schemas.md.mlip_md_equilibration import (
    MlipMdEquilibrationSimulationSettings,
)
from crystal.schemas.md.settings import (
    MdDynamicsSettings,
    MdEnsemble,
    MdPotentialSettings,
    MdRunSettings,
    MdThermostatSettings,
    MdTimeSettings,
)
from crystal.schemas.mlip_model_ref import MlipModelRef, QetModelRegistry
from crystal.structure.interfaces.builder import Interface


## 2. Set the input path and production parameters

This step selects the input interface and bundled model. Set `INTERFACE_PATH` to `DATA_DIR / "selected_interface.extxyz"` to use Notebook 05's selected geometry instead of the bundled starting interface.

This step sets a single temperature of 300 K, a time step of 1 fs, and a production time of 0.004 ps. This gives four MD steps. The thermostat time constant is 100 fs. The energy averaging window is 0.001 ps. These short intervals are for demonstration only.

The local run creates separate job directories below `results/md-runs/`. The final JSON file is overwritten after a successful run. No run flag is used.

In [ ]:
DATA_DIR = CRYSTAL_ROOT / "examples/results"
INTERFACE_PATH = CRYSTAL_ROOT / "examples/data/mock_interface.extxyz"
RUN_DIRECTORY = DATA_DIR / "md-runs"
RESULT_PATH = DATA_DIR / "work-of-adhesion.json"

TEMPERATURES_K = [300.0]
TIME_STEP_FS = 1.0
SIMULATION_TIME_PS = 0.004
THERMOSTAT_TIME_CONSTANT_FS = 100.0
ENERGY_AVERAGING_WINDOW_PS = 0.001

model = MlipModelRef(registry_model=QetModelRegistry.finetuned_linbosi)


## 3. Read the interface

This step reads the extxyz file and reconstructs the `Interface` object from its metadata. It normalises numeric metadata with `jsanitize` so that the workflow can store and reconstruct it. The per-atom `interface_label` values identify the film and substrate atoms.

This step prints the input path, atom count, and model path. The bundled interface is a starting geometry. It has not been relaxed or scientifically validated.

In [ ]:
if not INTERFACE_PATH.is_file():
    msg = f"No interface file at {INTERFACE_PATH}. Check the path or complete Notebook 05."
    raise FileNotFoundError(msg)
ase_atoms = read(INTERFACE_PATH.as_posix(), format="extxyz")
ase_atoms.info = jsanitize(ase_atoms.info, strict=True)
interface = Interface.from_structure(AseAtomsAdaptor.get_structure(ase_atoms))
print("Input interface:", INTERFACE_PATH)
print("Interface atoms:", len(interface))
print("MLIP model:", model.model_path)

## 4. Build the production-only MD jobs

This step builds one adhesion job for each temperature. Each job uses the same production settings for the top slab, bottom slab, and interface. The production ensemble is NVT. For temperature dependence, the same job can be run with different temperatures.

This step sets `interface_md_settings=[]` to skip the interface equilibration stages. The existing API uses `MlipMdEquilibrationSimulationSettings` for the production settings too. 

The empty `checks` list disables equilibration checks. A successful job does not prove thermal equilibrium. For scientific use, equilibrate the structures and test the production duration and energy averaging window.

The MD output directory remains unset. Each production job uses its own working directory so that its trajectory and log files do not overwrite another job's files.

In [ ]:
temperature_jobs: list[tuple[float, Job]] = []
for temperature in TEMPERATURES_K:
    production = MlipMdEquilibrationSimulationSettings(
        run_settings=MdRunSettings(
            time=MdTimeSettings(
                time_step_fs=TIME_STEP_FS,
                simulation_time_ps=SIMULATION_TIME_PS,
            ),
            dynamics=MdDynamicsSettings(
                ensemble=MdEnsemble.nvt,
                thermostat=MdThermostatSettings(
                    temperature_k=temperature,
                    time_constant_fs=THERMOSTAT_TIME_CONSTANT_FS,
                ),
            ),
            potential=MdPotentialSettings(model=model),
        ),
        checks=[],
        time_averaging_window=ENERGY_AVERAGING_WINDOW_PS,
    )
    job = MDAdhesionEnergyMaker(
        simulation_settings=MDAdhesionEnergySimulationSettings(
            interface_md_settings=[],
            production_md_setting=production,
        ),
    ).make(interface_structure=interface)
    temperature_jobs.append((temperature, job))
print("Adhesion jobs:", len(temperature_jobs))


## 5. Run the MD jobs locally

This step creates an in-memory JobStore and runs each adhesion job on this computer. The maker creates the slab and interface production branches during execution. Separate job directories keep their trajectories and logs apart.

The execution waits for each job to finish before starting the next temperature. The results remain available through `store` while this kernel is active. A kernel restart removes the in-memory records. The calculation files remain below `results/md-runs/`.

In [ ]:
RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)
store = JobStore(MemoryStore())
for temperature, job in temperature_jobs:
    print(f"Run production MD at {temperature:g} K")
    run_locally(
        job,
        store=store,
        root_dir=RUN_DIRECTORY,
        create_folders=True,
        ensure_success=True,
    )


## 6. Retrieve and report the work of adhesion

This step resolves each job's final output from the in-memory store. The outer job reference resolves the workflow created during execution. The existing maker combines the average slab and interface energies and divides the adhesion energy by the interface area.

This step prints the work of adhesion in eV/Å² and the interface area in Å². It saves the temperatures and full results to `results/work-of-adhesion.json`. The stored slab and interface energies are in eV. The JSON file is overwritten each time this cell runs.

Keep the calculation directories for inspection of the trajectories and energy logs. The four-step result is a workflow demonstration, not a converged adhesion value.

In [ ]:
records = []
for temperature, job in temperature_jobs:
    resolved_result = job.output.resolve(store)
    if resolved_result is None:
        msg = f"No adhesion result is available at {temperature:g} K. Inspect the MD jobs under {RUN_DIRECTORY}."
        raise RuntimeError(msg)
    result = MDAdhesionEnergyResult.model_validate(resolved_result)
    records.append({"temperature_k": temperature, "result": result})
    print(
        f"{temperature:g} K: {result.work_of_adhesion_per_area:.6f} eV/Å²; "
        f"area={result.interface_area:.3f} Å²"
    )
dumpfn(records, RESULT_PATH)
print("Saved adhesion results:", RESULT_PATH)
